# Part 1

### Configuration & Catalog Setup

In [0]:
dbutils.widgets.text("catalog", "de_assessment_dev")
CATALOG = dbutils.widgets.get("catalog")

STORAGE_ACCOUNT = "deassessmentd06fabcc"
RAW_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.sql(f"USE CATALOG {CATALOG}")

### Fetch Shows

In [0]:
import time, requests

def fetch_with_retry(url, max_retries=4, timeout=30):
    """GET with exponential backoff — handles TVMaze 429 rate limits and timeouts."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.exceptions.Timeout:
            wait = 2 ** attempt
            print(f"Timeout on {url}. Waiting {wait}s before retry {attempt+1}/{max_retries}")
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise Exception(f"All {max_retries} attempts failed for {url}: {e}")
            time.sleep(2 ** attempt)
    raise Exception(f"Exhausted {max_retries} retries for {url}")

In [0]:
import json
import pandas as pd

response = fetch_with_retry("https://api.tvmaze.com/shows")
shows = response.json()

shows_pdf = pd.DataFrame(shows)
for col_name in shows_pdf.columns:
    if shows_pdf[col_name].dtype == object:
        shows_pdf[col_name] = shows_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

shows_df = spark.createDataFrame(shows_pdf)
print(f"Fetched {len(shows)} shows")

In [0]:
# print(shows_df.schema)

# shows_df.printSchema()
# shows_df.show(5, truncate=False)
# print(shows_df.count())
# print(shows_df.columns)
print(json.dumps(shows[1], indent=2))
# shows_df.select("id", "name", "network").show(5, truncate=False)

### Write shows to ADLS Gen 2 Storage

In [0]:
# Write json to raw container using the external location name (Unity Catalog routes auth correctly)
shows_df.write.mode("overwrite").json(f"{RAW_PATH}/shows/")

### Save as shows Bronze Delta Tables

In [0]:
# Read back and save as Bronze Delta table
bronze_shows = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/shows/")

bronze_shows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_shows")

print("bronze_shows created:", bronze_shows.count(), "rows")

### Fetch Episodes & Cast

In [0]:
import time

show_ids = [show["id"] for show in shows]

# Fetch episodes for all shows
episodes = []
for i, show_id in enumerate(show_ids):
    try:
        response = fetch_with_retry(f"https://api.tvmaze.com/shows/{show_id}/episodes")
        for ep in response.json():
            ep["show_id"] = show_id
            episodes.append(ep)
    except Exception as e:
        print(f"Warning: failed to fetch episodes for show {show_id}: {e}")
    if (i + 1) % 20 == 0:
        print(f"Fetched episodes for {i + 1}/{len(show_ids)} shows")
        time.sleep(1)  # Rate limit courtesy pause

# Fetch cast for all shows
cast = []
for i, show_id in enumerate(show_ids):
    try:
        response = fetch_with_retry(f"https://api.tvmaze.com/shows/{show_id}/cast")
        for member in response.json():
            member["show_id"] = show_id
            cast.append(member)
    except Exception as e:
        print(f"Warning: failed to fetch cast for show {show_id}: {e}")
    if (i + 1) % 20 == 0:
        print(f"Fetched cast for {i + 1}/{len(show_ids)} shows")
        time.sleep(1)  # Rate limit courtesy pause

print(f"Episodes: {len(episodes)}, Cast: {len(cast)}")

### Write Episodes & Cast to ADLS

In [0]:
# Flatten complex fields for Spark
episodes_pdf = pd.DataFrame(episodes)
for col_name in episodes_pdf.columns:
    if episodes_pdf[col_name].dtype == object:
        episodes_pdf[col_name] = episodes_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

cast_pdf = pd.DataFrame(cast)
for col_name in cast_pdf.columns:
    if cast_pdf[col_name].dtype == object:
        cast_pdf[col_name] = cast_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

episodes_df = spark.createDataFrame(episodes_pdf)
cast_df = spark.createDataFrame(cast_pdf)

episodes_df.write.mode("overwrite").json(f"{RAW_PATH}/episodes/")
cast_df.write.mode("overwrite").json(f"{RAW_PATH}/cast/")

print(f"Written {episodes_df.count()} episodes and {cast_df.count()} cast members to raw")

### Save Episodes & Cast as Bronze Delta Table

In [0]:
bronze_episodes = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/episodes/")
bronze_cast = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/cast/")

bronze_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_episodes")

bronze_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_cast")

print("bronze_episodes:", bronze_episodes.count(), "rows")
print("bronze_cast:", bronze_cast.count(), "rows")

### Grant access to the Compliance Team

In [0]:
tables = ["bronze_shows", "bronze_episodes", "bronze_cast"]

try:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `Compliance-Team`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.bronze TO `Compliance-Team`")
    for table in tables:
        spark.sql(f"GRANT SELECT ON TABLE {CATALOG}.bronze.{table} TO `Compliance-Team`")
    print("Compliance-Team grants applied successfully")
except Exception as e:
    print(f"Warning: could not apply Compliance-Team grants: {e}")

### Explanation of schema evolution approach.

Bronze ingests raw TVmaze API payloads without enforcing a schema. Delta writes use overwriteSchema=True, so the table schema is fully replaced on each run to exactly match the current DataFrame. Since bronze does a complete re-fetch every run, the schema always reflects the latest API response. No data is ever rejected at this layer. 

In [0]:
import json

counts = {
    "bronze_shows":    spark.table(f"{CATALOG}.bronze.bronze_shows").count(),
    "bronze_episodes": spark.table(f"{CATALOG}.bronze.bronze_episodes").count(),
    "bronze_cast":     spark.table(f"{CATALOG}.bronze.bronze_cast").count(),
}

failed = [k for k, v in counts.items() if v == 0]
if failed:
    msg = f"Bronze validation FAILED — empty tables: {failed}"
    print(msg)
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "reason": msg, "counts": counts}))

print("Bronze validation passed:", counts)
dbutils.notebook.exit(json.dumps({"status": "OK", "counts": counts}))